In [2]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(ROOT))

from src.paths import CLEAN_DATA
from src.benchmark import analytic_benchmark
from src.fully_connected import train_model
from src.metrics import gain
from src.helper import make_run_dir, save_results

## Load data

In [3]:
df_train = pd.read_parquet(CLEAN_DATA / 'rand_A_train.parquet')
df_val   = pd.read_parquet(CLEAN_DATA / 'rand_A_val.parquet')
df_test  = pd.read_parquet(CLEAN_DATA / 'rand_A_test.parquet')

## Analytic benchmark

In [4]:
hw = analytic_benchmark(
    df_train['delta'].values, df_train['T'].values, df_train['spy_ret'].values, df_train['d_iv'].values,
    df_val['delta'].values,   df_val['T'].values,   df_val['spy_ret'].values,   df_val['d_iv'].values,
    df_test['delta'].values,  df_test['T'].values,   df_test['spy_ret'].values,  df_test['d_iv'].values,
)

Analytic Benchmark — SSE: 115.3004  RMSE: 0.017445
Coefficients: a = -0.137149, b = -0.079070, c = -0.056295


## 3-Feature & 3-Feature ANN 

In [5]:
# 3-Feature ANN
features_3f = ['delta', 'T', 'spy_ret']

result_3f = train_model(
    df_train[features_3f].values, df_train['d_iv'].values,
    df_val[features_3f].values,   df_val['d_iv'].values,
    df_test[features_3f].values,  df_test['d_iv'].values,
    desc="ANN 3F",
)

# 4-Feature ANN
features_4f = ['delta', 'T', 'spy_ret', 'vix_lag']

result_4f = train_model(
    df_train[features_4f].values, df_train['d_iv'].values,
    df_val[features_4f].values,   df_val['d_iv'].values,
    df_test[features_4f].values,  df_test['d_iv'].values,
    desc="ANN 4F",
)

ANN 3F:   0%|          | 0/2 epochs [00:00<?, ?epoch/s]  


Test — SSE: 107.0089  RMSE: 0.016806


ANN 4F:   0%|          | 0/2 epochs [00:00<?, ?epoch/s]  


Test — SSE: 100.3355  RMSE: 0.016274


In [6]:
out = make_run_dir("1.0-ann")
save_results(result_3f, "ann_3f", out)
save_results(result_4f, "ann_4f", out)
print(f"\nResults saved to: {out}")

  saved → /Users/arj/Desktop/York/Schulich/Semester-2/MATH 6912/Math6912-Project/IV Project/output/1.0-ann/run-01/ann_3f.*
  saved → /Users/arj/Desktop/York/Schulich/Semester-2/MATH 6912/Math6912-Project/IV Project/output/1.0-ann/run-01/ann_4f.*

Results saved to: /Users/arj/Desktop/York/Schulich/Semester-2/MATH 6912/Math6912-Project/IV Project/output/1.0-ann/run-01


## Results

In [7]:
gain_3f = gain(result_3f['sse'], hw['sse']) * 100
gain_4f = gain(result_4f['sse'], hw['sse']) * 100
gain_4f_vs_3f = gain(result_4f['sse'], result_3f['sse']) * 100

print(f'\n{'='*40}')
print(f'Analytic  — SSE: {hw['sse']:.4f}')
print(f'ANN 3F    — SSE: {result_3f['sse']:.4f}  Gain vs Analytic: {gain_3f:.2f}%')
print(f'ANN 4F    — SSE: {result_4f['sse']:.4f}  Gain vs Analytic: {gain_4f:.2f}%  Gain vs 3F: {gain_4f_vs_3f:.2f}%')
print(f'{'='*40}')


Analytic  — SSE: 115.3004
ANN 3F    — SSE: 107.0089  Gain vs Analytic: 7.19%
ANN 4F    — SSE: 100.3355  Gain vs Analytic: 12.98%  Gain vs 3F: 6.24%
